# Portfolio Chat Agent — RAG Pipeline

This notebook builds a full **Retrieval-Augmented Generation (RAG)** pipeline using LangChain and Gemini.
This is the prototyping notebook I used to develop a working chat agent that answers questions about my CV, thesis work, and professional experience.

**Pipeline overview:**
```
Your documents (PDF/DOCX)
       ↓  Document Loaders
   Raw text pages
       ↓  Text Splitter
   Smaller chunks
       ↓  Embedding Model (Gemini)
   Vectors stored in ChromaDB
       ↓  Retriever
   Relevant chunks for a query
       ↓  LLM (Gemini) + Prompt
   Final answer
```

## 1. Setup — imports and environment

We load the Gemini API key from `.env` using `python-dotenv`, then import everything we need from LangChain.

In [2]:
! pip install -r requirements.txt

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env and injects GOOGLE_API_KEY into os.environ

api_key = os.getenv('GOOGLE_API_KEY')
assert api_key, 'GOOGLE_API_KEY not found — check your .env file'
print('API key loaded:', api_key[:8], '...')

API key loaded: AIzaSyBe ...


In [2]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print('All imports OK')

C:\Users\manum\AppData\Local\Temp\ipykernel_30372\2825302434.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader


All imports OK


## 2. Document Loading

LangChain has loaders for almost every file format. Each loader returns a list of **`Document`** objects.
A `Document` has two fields:
- `page_content` — the raw text
- `metadata` — dict with source, page number, etc.

We load all files from the `data/` folder automatically.

In [ ]:
DATA_DIR = 'data - copia'

from langchain_community.document_loaders import TextLoader
import re

def load_documents(data_dir):
    docs = []
    for filename in sorted(os.listdir(data_dir)):
        filepath = os.path.join(data_dir, filename)
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(filepath)
            docs.extend(loader.load())
            print(f'  Loaded PDF:  {filename}')
        elif filename.endswith('.md'):
            loader = TextLoader(filepath, encoding='utf-8')
            docs.extend(loader.load())
            print(f'  Loaded MD:   {filename}')
        elif filename.endswith('.docx'):
            # Skipped — replaced by cleaned .md equivalents
            print(f'  Skipped DOCX (use .md version): {filename}')
    return docs

raw_docs = load_documents(DATA_DIR)
print(f'\nTotal pages/documents loaded: {len(raw_docs)}')

In [6]:
sample = raw_docs[0]
print('--- metadata ---')
print(sample.metadata)
print('\n--- first 500 chars of content ---')
print(sample.page_content[:500])

--- metadata ---
{'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Manuel Mezo CV - Applied AI 2026', 'source': 'data\\Manuel Mezo CV - Applied AI 2026.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}

--- first 500 chars of content ---
Manuel
 
Mezo
 Madrid,  Spain  |  +34  652257366  |  manumezog@gmail.com Github |  Project  portfolio |  LinkedIn 
 PROFILE   
Engineer  and  applied  AI  builder  with  an  8+  year  track  record  of  solving  multi-million-dollar  operational  
challenges
 
at
 
Amazon
 
and
 
McKinsey,
 
now
 
independently
 
shipping
 
production
 
AI
 
systems:
 
LLM
 
agents,
 
computer
 
vision
 
pipelines,
 
voice-first
 
interfaces,
 
and
 
multi-agent
 
simulations.
 
Combines
 
deep
 
operational
 
i


## 3. Text Splitting

LLMs and embedding models have context limits, and large pages contain multiple topics. We split each document into smaller **chunks**.

`RecursiveCharacterTextSplitter` tries to split on paragraphs first, then sentences, then words — so chunks stay semantically coherent.

Key parameters:
- `chunk_size` — max characters per chunk (~500 is a good default for dense PDFs)
- `chunk_overlap` — characters shared between consecutive chunks (prevents cutting a sentence mid-thought)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,   # larger than before — narrative text needs more context per chunk
    chunk_overlap=150,
    separators=['\n\n', '\n', ' ', ''],
)

chunks = splitter.split_documents(raw_docs)
chunks = [c for c in chunks if c.page_content.strip()]  # drop empty chunks
print(f'Raw pages/docs: {len(raw_docs)}  ->  Chunks after splitting: {len(chunks)}')

In [9]:
sample_chunk = chunks[5]
print('metadata:', sample_chunk.metadata)
print('\ncontent:')
print(sample_chunk.page_content)

metadata: {'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Manuel Mezo CV - Applied AI 2026', 'source': 'data\\Manuel Mezo CV - Applied AI 2026.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}

content:
surfacing
 
hidden
 
failure
 
modes
 
in
 
supply
 
chain
 
planning
 
logic.
 ●  WMS  with  Camera  Vision:  Built  a  mobile-first  warehouse  management  system  using  on-device  
camera
 
for
 
barcode
 
scanning
 
and
 
real-time
 
inventory
 
sync
 
via
 
Firebase.
  
COMPUTER  VISION  AND  ROBOTICS  PROJECTS   
●  Custom  YOLOv8  Object  Tracking  &  Spatial  Logic:  Engineered  a  custom  detection  pipeline  
using
 
an
 
annotated
 
dataset
 
in
 
Roboflow.
 
Fine-tuned
 
a


In [5]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key,
)

test_vector = embeddings.embed_query("What is Manuel's work experience?")
print(f"Embedding dimension: {len(test_vector)}")
print(f"First 5 values: {test_vector[:5]}")

Embedding dimension: 3072
First 5 values: [-0.0079705445, 0.02044252, 0.011608609, -0.059756465, -0.007417933]


In [7]:
import uuid, shutil

CHROMA_DIR = 'chroma_db'

# Remove any previous partial store
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

texts   = [c.page_content for c in chunks]
metas   = [c.metadata     for c in chunks]
ids     = [str(uuid.uuid4()) for _ in chunks]

# Embed all chunks first, then add to Chroma directly.
# This bypasses a batching bug in langchain_community Chroma.
print(f'Embedding {len(chunks)} chunks — this takes ~30s...')
vectors = embeddings.embed_documents(texts)

vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
vectorstore._collection.add(documents=texts, embeddings=vectors, metadatas=metas, ids=ids)

print(f'Vector store created. Collection size: {vectorstore._collection.count()} chunks')

Embedding 318 chunks — this takes ~30s...
Vector store created. Collection size: 318 chunks


In [ ]:
# On subsequent runs reload from disk to skip re-embedding:
# vectorstore = Chroma(
#     persist_directory=CHROMA_DIR,
#     embedding_function=embeddings,
# )
# print('Loaded from disk:', vectorstore._collection.count(), 'chunks')

## 6. Retriever — searching by similarity

A **retriever** wraps the vector store and, given a query, returns the `k` most similar chunks. Internally: it embeds the query, finds nearest vectors, and returns those documents.

Let's test it before wiring up the LLM.

In [8]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4},  # return top 4 most relevant chunks
)

query = "What did Manuel study?"
relevant_chunks = retriever.invoke(query)

print(f"Query: '{query}'")
print(f'Found {len(relevant_chunks)} relevant chunks:\n')
for i, doc in enumerate(relevant_chunks):
    print(f"--- Chunk {i+1} | source: {doc.metadata.get('source', 'unknown')} ---")
    print(doc.page_content[:300])
    print()

Query: 'What did Manuel study?'
Found 4 relevant chunks:

--- Chunk 1 | source: data\Profesional experience project detailed examples.docx ---
Additionally he handled with “brio” this cross-functional project until financial and implementation approval identifying the tasks, resources and support needed to enable the solution (finance, procurement, TPM, ops). 

Ultimately, he proposed a pilot plan that optimized value, effort and implement

--- Chunk 2 | source: data\Profesional experience project detailed examples.docx ---
Additionally, he has incorporated new metrics (DEA-EF, SSP), and adjusted existing ones to newly released versions (late slam, concessions) in effort to replicate FC reporting practices.

Manuel showed ownership by driving the complete process for EF integration into existing FC reporting and analys

--- Chunk 3 | source: data\Profesional experience project detailed examples.docx ---
A:  

Manuel arranged and conducted thorough onsite IRDR process audits (dive deep)

## 7. RAG Chain

Now we connect retrieval to the LLM. The chain does:
1. Take the user question
2. Retrieve relevant chunks from ChromaDB
3. Format a prompt with those chunks as context
4. Send to Gemini and return the answer

LangChain's `|` pipe operator chains these steps together — this is the **LCEL** (LangChain Expression Language). Think of it like a Unix pipe: each step's output feeds the next step's input.

In [17]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',  # swap for gemini-1.5-pro for better quality
    google_api_key=api_key,
    temperature=0.2,  # low = more factual, less creative
)

print('LLM ready:', llm.model)

LLM ready: gemini-2.5-flash


In [18]:
SYSTEM_PROMPT = (
    "You are a helpful assistant embedded in Manuel Mezo's personal portfolio website.\n"
    "Answer questions about Manuel's education, professional experience, skills, and projects.\n"
    "Base your answers ONLY on the context provided below — do not invent information.\n"
    "If the context does not contain enough information to answer, say so honestly.\n"
    "Keep answers concise and professional.\n\n"
    "Context:\n{context}\n"
)

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', '{question}'),
])

print('Prompt template ready')

Prompt template ready


In [19]:
def format_docs(docs):
    return '\n\n'.join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# LCEL chain — how the pipe works:
#   {'context': retriever | format_docs, 'question': RunnablePassthrough()}
#     builds a dict: 'context' = retrieved+formatted chunks, 'question' = user input unchanged
#   | prompt          fills the prompt template with that dict
#   | llm             sends the filled prompt to Gemini
#   | StrOutputParser extracts plain text from the LLM response object

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG chain assembled')

RAG chain assembled


## 8. Try it out!

Run the cells below with different questions to test the agent.
Each call goes through the full pipeline: embed query → retrieve chunks → generate answer.

In [20]:
def ask(question):
    print(f'Q: {question}')
    print('-' * 60)
    answer = rag_chain.invoke(question)
    print(answer)
    print()

ask("What is Manuel's educational background?")

Q: What is Manuel's educational background?
------------------------------------------------------------
Manuel has an MSc in Aerospace Engineering from Universidad Carlos III de Madrid and TU Delft (Exchange), and a BSc in Aerospace Engineering & BSc in Mechanical Engineering from Universidad de León.

He also holds:
*   A Certificate of Proficiency: ROS2 Basics (Python) from The Construct Robotics Institute (2026).
*   A Python for data science certification by Microsoft (2016).



In [21]:
ask('What professional projects has Manuel worked on?')

Q: What professional projects has Manuel worked on?
------------------------------------------------------------
Manuel has worked on several professional projects, including:
*   A cross-functional project that he handled until financial and implementation approval.
*   Proposing a pilot plan that optimized value, effort, and implementation time.
*   Driving the implementation phase in the pilot node XDEQ in Germany.
*   Driving the complete process for EF integration into existing FC reporting and analysis tools like Stay Clean and Minerva.
*   Developing the Technical BI & Data management structure for the whole EF organization.



In [22]:
ask("What was Manuel's master thesis about?")

Q: What was Manuel's master thesis about?
------------------------------------------------------------
Manuel Mezo's Master of Science thesis was about "Open- and closed-loop Doppler data modelling for spacecraft tracking."



In [23]:
ask("What are Manuel's main technical skills?")

Q: What are Manuel's main technical skills?
------------------------------------------------------------
Manuel's main technical skills include:
*   **AI & LLM:** LLM APIs (Claude, Gemini 2.5, GPT-4), multi-agent systems, voice agent stacks (Retell.ai, Voiceflow), prompt engineering, structured outputs.
*   **Engineering:** Python, cloud infrastructure, ML fine-tuning, and real-time deployment.



In [24]:
# Try your own question here
ask("Tell me about Manuel's engineering degree thesis")

Q: Tell me about Manuel's engineering degree thesis
------------------------------------------------------------
Manuel Mezo completed two engineering degree theses:

1.  **Mechanical Engineering Thesis:** "Diseño preliminar del Sistema de Potencia Eléctrica de un vehículo espacial para una misión a la EEI" (Preliminary design of the Electrical Power System of a spacecraft for a mission to the ISS), completed on September 22, 2016.
2.  **Bachelor's Thesis:** "Estudio de viabilidad técnica para la instalación de un banco de ensayos para pequeños turborreactores en la Universidad de León" (Technical feasibility study for the installation of a test bench for small turbojets at the University of León), completed on July 3, 2015.



In [25]:
# Try your own question here
ask("Make a summary of his mastersthesis")

Q: Make a summary of his mastersthesis
------------------------------------------------------------
Manuel Mezo's Master's thesis is titled "Open- and closed-loop Doppler data modelling for spacecraft tracking." The thesis covers the framework of spacecraft tracking and Doppler data, including research questions and conversion models. It details an analysis strategy involving scenarios, sensitivity analysis, figures of merit, and test cases. The development section focuses on validation, while the results cover sensitivity to inherent parameters and geometry, and a comparison of models, leading to conclusions.



## 9. What's next?

The pipeline is working. Natural next steps:

1. **Add more documents** — drop any PDF/DOCX into `data/` and re-run from section 2
2. **Improve the prompt** — tune tone, language (English/Spanish), add more persona details
3. **Add chat history** — use `RunnableWithMessageHistory` so the agent handles follow-up questions
4. **Wrap in FastAPI** — expose `rag_chain.invoke()` as a `POST /chat` endpoint
5. **Connect to your portfolio** — call the API from a chat widget in your frontend

- **To keep learning LangChain** → step 3 (chat history introduces memory concepts)
- **To get it live on your portfolio ASAP** → step 4 + 5

## 10. Chat History — Conversational RAG

So far every question is independent. If the user asks *"What was his thesis about?"* after asking about education,
the agent has no memory of what "his" refers to.

**The fix: two chained steps**

```
User question + chat history
        ↓  Contextualize chain (LLM)
  Standalone rephrased question   ← "What was Manuel's master thesis about?"
        ↓  RAG chain (retriever + LLM)
  Final answer
```

**Key concept — `RunnableWithMessageHistory`**
It wraps any chain and:
1. Loads the stored history for the current session before each call
2. Appends the new human message and AI response after each call
3. Keeps separate histories per `session_id` so multiple users don't mix


In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# --- Step 1: Contextualize question ---
# This prompt tells the LLM: given the chat history, rewrite the user question
# as a self-contained question (no pronouns, no references to previous turns).
# If the question is already standalone, return it unchanged.
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, "
     "rewrite it as a standalone question. "
     "Do NOT answer — only reformulate if needed, otherwise return as-is."),
    MessagesPlaceholder("chat_history"),   # previous turns injected here
    ("human", "{input}"),
])

# history_aware_retriever = contextualize chain + retriever
# It rephrases the question, then retrieves chunks using the rephrased version
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)

# --- Step 2: Answer chain ---
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant on Manuel Mezo's portfolio website. "
     "Answer questions about Manuel's education, experience, skills, and projects. "
     "Use ONLY the context below — do not invent information. "
     "Be concise and professional.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# create_stuff_documents_chain: takes retrieved docs, formats them, sends to LLM
question_answer_chain = create_stuff_documents_chain(llm, answer_prompt)

# combine both: history-aware retrieval + answering
rag_chain_with_history = create_retrieval_chain(history_aware_retriever, question_answer_chain)
print("Conversational RAG chain ready")

In [ ]:
# In-memory store: session_id -> ChatMessageHistory
# In production you'd persist this to Redis or a database
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# RunnableWithMessageHistory wraps the chain and handles history automatically.
# input_messages_key: which key in the input dict is the user question
# history_messages_key: which key the chain expects for prior messages
# output_messages_key: where the chain puts its answer (for history storage)
conversational_rag = RunnableWithMessageHistory(
    rag_chain_with_history,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)
print("RunnableWithMessageHistory ready")

In [ ]:
def chat(question, session_id="demo"):
    result = conversational_rag.invoke(
        {"input": question},
        config={"configurable": {"session_id": session_id}},
    )
    return result["answer"]

# Test a multi-turn conversation — notice the follow-up uses "it" and "his"
# and the agent still answers correctly because history gives it context
print("Turn 1:", chat("What did Manuel study?"))
print()
print("Turn 2:", chat("What was his thesis about?"))   # 'his' resolved via history
print()
print("Turn 3:", chat("What tools did he use for it?"))  # 'it' = the thesis